# Ablaciones y paridad de exportación nativa: estado del arte

**Nivel:** 3  
**Requisitos previos:** Ablaciones y paridad de exportación nativa nivel 1  
**Tiempo de lectura:** 25 minutos

## Objetivos

El objetivo es explicar el problema y sus datos, inspeccionar y, en niveles superiores, ejecutar el método publicado en su tarea emparejada. Una evaluación completa debe comprobar si cada componente propuesto contribuye y si las exportaciones optimizadas del modelo preservan el puntuador de referencia. La coincidencia de puntajes es evidencia numérica; no demuestra decisiones más seguras, mejores resultados de tarea ni generalización.

## Preparación

Las versiones siguientes están fijadas en `notebooks/requirements.txt` y también constan en los metadatos de este notebook.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Datos

El notebook lee artefactos comprometidos del producto. Propietario: Neuraxis. Repositorio: https://github.com/fsantibanezleal/CAOS_Neuraxis. Licencia: Apache-2.0. El JSON es un artefacto experimental propio y congelado; este notebook no redistribuye datos de benchmarks externos.

Este notebook lee `data/derived/index.json` y las trazas listadas. La ejecución usa un espacio aislado de caso Reflexmesh. Para métodos aprendidos, descarga los activos públicos exactos con `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000` y define `NEURAXIS_CHECKPOINTS` hacia esa carpeta. El manifiesto de publicación verifica los hashes. M09/M10 se leen desde trazas congeladas, salvo que `NEURAXIS_NOTEBOOK_MODEL` indique explícitamente un modelo Ollama local configurado.

Artefactos de esta lección:

- `data/derived/index.json`
- `data/derived/research.json`
- `docs/evaluation/corrected-results.json`
- `data/derived/episodes/M03-C07-nominal-40000.json`
- `data/derived/episodes/M04-C07-nominal-40000.json`
- `data/derived/episodes/M11-C07-nominal-40000.json`
- `data/derived/episodes/M12-C07-nominal-40000.json`

## Método

Audita el digest del índice canónico frente al informe de investigación, compara las cuatro ablaciones emparejadas de M12 con el controlador completo y luego inspecciona los vectores de exportación nativa M03/M04, cobertura de umbrales, tolerancia de error y alcance de latencia en caliente.

Para el resultado emparejado \(Y\), el contraste de ablación es \(\Delta=Y_{completo}-Y_{ablacionado}\); las semillas emparejadas y las plantillas fijas determinan la unidad de incertidumbre. La paridad de exportación es \(\max_i |p_i^{nativo}-p_i^{referencia}|\le\epsilon\), un criterio numérico separado.

### Comprueba las versiones fijadas del entorno.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verifica la auditoría y los bytes de cada traza antes de interpretar resultados.

In [2]:
# Lee la evidencia comprometida y usa la API publicada; nunca define otra política.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M03', 'M04', 'M11', 'M12']
sample_case = 'C07'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C07 | seed: 40000
Methods: M03, M04, M11, M12


### Inspecciona las decisiones y resultados originales

In [3]:
# Resume la primera acción registrada, los candidatos autorizados y el resultado del oráculo independiente.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M03 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M04 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M11 | action= a-cancel | reason= learned_incremental_benefit_gate | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M12 | action= a-cancel | reason= urgent_cancellation_during_deliberation | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


### Ejecuta las políticas publicadas y compara la corrida

In [4]:
# Los resultados de planificación se leen de las trazas congeladas; no hay llamada implícita a un LLM.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M03 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M04 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M11 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M12 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}


### Compara la secuencia completa y no solo la primera acción

In [5]:
# La igualdad de secuencia es diagnóstica; la verificación independiente decide el éxito.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M03 a-cancel | terminal= succeeded | verified= 1
M04 a-cancel | terminal= succeeded | verified= 1
M11 a-cancel | terminal= succeeded | verified= 1
M12 a-cancel | terminal= succeeded | verified= 1


### Recalcula resultados por variante desde todas las trazas del caso

In [6]:
# Lee todas las semillas del caso elegido y conserva cada episodio fallido en el denominador.
from collections import defaultdict
groups = defaultdict(list)
for row in index['episodes']:
    if row['case_id'] not in (sample_case,) or row['method_id'] not in selected_methods:
        continue
    raw = (ROOT / 'data/derived' / row['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row['sha256']
    episode = json.loads(raw)
    groups[(row['method_id'], row['variant'])].append(int(bool(episode['metrics']['task_success'])))
for (method_id, variant), values in sorted(groups.items()):
    print(method_id, variant, 'successes=', sum(values), 'episodes=', len(values), 'rate=', round(sum(values)/len(values), 3))

M03 boundary successes= 10 episodes= 10 rate= 1.0
M03 delayed_observation successes= 10 episodes= 10 rate= 1.0
M03 duplicate_event successes= 10 episodes= 10 rate= 1.0
M03 nominal successes= 10 episodes= 10 rate= 1.0
M03 stale_conflict successes= 10 episodes= 10 rate= 1.0
M03 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M04 boundary successes= 10 episodes= 10 rate= 1.0
M04 delayed_observation successes= 10 episodes= 10 rate= 1.0
M04 duplicate_event successes= 10 episodes= 10 rate= 1.0
M04 nominal successes= 10 episodes= 10 rate= 1.0
M04 stale_conflict successes= 10 episodes= 10 rate= 1.0
M04 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M11 boundary successes= 10 episodes= 10 rate= 1.0
M11 delayed_observation successes= 10 episodes= 10 rate= 1.0
M11 duplicate_event successes= 10 episodes= 10 rate= 1.0
M11 nominal successes= 10 episodes= 10 rate= 1.0
M11 stale_conflict successes= 10 episodes= 10 rate= 1.0
M11 unavailable_dependency successes= 10 episodes= 10

### Visualiza el estrés por variante sin mezclar familias

In [7]:
# Cada punto resume las semillas de una sola variante del caso, no el benchmark completo.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
variants = sorted(set(key[1] for key in groups))
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for method_id in selected_methods:
    rates = [sum(groups[(method_id, variant)])/len(groups[(method_id, variant)]) if groups[(method_id, variant)] else float('nan') for variant in variants]
    ax.plot(variants, rates, marker='o', label=method_id)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel('Tasa de éxito verificado')
ax.set_title(sample_case + ' por variante')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=.25)
ax.legend()
display(fig)
plt.close(fig)

<Figure size 900x450 with 1 Axes>

### Recalcula las ablaciones emparejadas

In [8]:
# La investigación está ligada al digest del índice canónico y conserva emparejamiento por celda.
research = json.loads((ROOT / 'data/derived/research.json').read_text(encoding='utf-8'))
assert research['core']['index_sha256'] == hashlib.sha256(index_bytes).hexdigest()
full = next(row for row in research['core']['methods'] if row['method_id']=='M12')
print('Full M12:', full['successes'], '/', full['episodes'], '| rate=', full['success_rate'])
for name, result in research['ablations'].items():
    ablated = next(row for row in result['methods'] if row['method_id']=='M12')
    paired = result['paired_full_minus_ablation']
    assert paired['paired_cells'] == ablated['episodes'] == 1200
    assert ablated['successes'] == full['successes']
    print(name, '| successes=', ablated['successes'], '/', ablated['episodes'], '| paired delta=', paired['success_difference_left_minus_right'], '| seed bootstrap95=', paired['paired_seed_bootstrap95'], '| full-minus-ablation wall ms=', round(paired['mean_wall_difference_ms_left_minus_right'],3), '| scope=', paired['scope'])
assert len(research['ablations']) == 4

Full M12: 1160 / 1200 | rate= 0.9666666666666667
no_memory | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 2.588 | scope= designed shared templates; small seed-group population; compute budgets are not equalized
no_effects | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 39.28 | scope= designed shared templates; small seed-group population; compute budgets are not equalized
no_deferral | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 131.89 | scope= designed shared templates; small seed-group population; compute budgets are not equalized
no_invalidation | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 5.0 | scope= designed shared templates; small seed-group population; compute budgets are not equalized


### Audita la paridad numérica de M03/M04

In [9]:
# La paridad numérica no demuestra seguridad, calibración de decisiones ni generalización.
for method_id, result in research['native_parity']['results'].items():
    assert result['passed'] and result['vectors'] == 484 and result['split_thresholds'] == 76
    assert result['max_abs_probability_error'] <= result['tolerance']
    timing = result['warmed_latency_ms']
    print(method_id, '| vectors=',result['vectors'],'| thresholds=',result['split_thresholds'],'| max error=',result['max_abs_probability_error'],'<=',result['tolerance'])
    print('  warmed p50/p95 ms native=',timing['native_cached']['50'],timing['native_cached']['95'],'| Python=',timing['python_reference']['50'],timing['python_reference']['95'])
    print('  scope:',result['warmed_timing_scope'])
print(research['native_parity']['note'])

M03 | vectors= 484 | thresholds= 76 | max error= 3.1308289294429414e-14 <= 1e-05
  warmed p50/p95 ms native= 0.009 0.0118 | Python= 0.10905000000000001 0.1334849999999999
  scope: Native: packed feature serialization + cached Rust scoring + return list; Python: fitted model predict + Platt; both exclude shared feature construction and effect execution
M04 | vectors= 484 | thresholds= 76 | max error= 3.7825163745619506e-07 <= 1e-05
  warmed p50/p95 ms native= 0.0185 0.0231 | Python= 0.2917 0.38110499999999975
  scope: Native: packed feature serialization + cached Rust scoring + return list; Python: fitted model predict + Platt; both exclude shared feature construction and effect execution
Numerical export parity, not outcome generalization or safety calibration evidence


## Estado del arte

XGBoost y PPO son familias de métodos establecidas; esta lección evalúa la implementación publicada de Neuraxis y su ruta de exportación, sin afirmar un algoritmo de aprendizaje nuevo. Las ablaciones y las comparaciones de implementación de referencia frente a optimizada son evidencia necesaria, pero ninguna demuestra por sí sola un avance del estado del arte.

## Resultados

Las cuatro ablaciones conservan los mismos 1.160/1.200 éxitos de tarea, así que este experimento no muestra contribución al éxito por memoria, efectos, aplazamiento o invalidación. Los presupuestos de ejecución difieren entre variantes, y las latencias medidas excluyen la construcción de características en Python. La paridad numérica no calibra la seguridad de la política.

Este paso de autoría no compromete salidas generadas; `scripts/run_notebooks.py` las ejecuta y guarda desde un kernel nuevo.

## Resumen

El resultado negativo de ablación importa: los componentes propuestos no cambian el éxito en esta suite saturada. Las exportaciones nativas cumplen la tolerancia numérica registrada, lo que respalda solo consistencia de implementación.

## Referencias

- [XGB](https://arxiv.org/abs/1603.02754)
- [PPO](https://arxiv.org/abs/1707.06347)